# EXPT NO: 7 — IMPLEMENTATION OF A SIMPLE AUTOENCODER FOR DIMENSIONALITY REDUCTION AND FEATURE LEARNING

**Name:** NIVASRI T  
**Roll No:** 24BAD081  
**Date:** 

---
## AIM
To implement and train a simple Autoencoder for dimensionality reduction and feature learning by compressing input data into a lower-dimensional representation and reconstructing the original data from the learned features.

## SOFTWARE REQUIREMENTS
Python, TensorFlow / Keras, Google Colab, GitHub, NumPy, Matplotlib, Pandas, Scikit-learn

In [ ]:
# ---------------------------------------------------------------
# Experiment 7 : Simple Autoencoder (MNIST)
# NAME    : NIVASRI T
# ROLL NO : 24BAD081
# ---------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error

SEED = 81
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Name    : NIVASRI T")
print("Roll No : 24BAD081")
print("TensorFlow version:", tf.__version__)

---
## PART A: DATASET PREPARATION

In [ ]:
# ---- A.1 Load the MNIST dataset ---- | Roll No: 24BAD081 | Name: NIVASRI T
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training set :", x_train_full.shape, x_train_full.dtype)
print("Test set     :", x_test.shape, x_test.dtype)
print("Label dtype  :", y_train_full.dtype)
print("Pixel range  :", x_train_full.min(), "to", x_train_full.max())

In [ ]:
# ---- A.2 Explore the dataset: sample images + class distribution ----
fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_full[i], cmap='gray'); ax.set_title(str(y_train_full[i]), fontsize=9); ax.axis('off')
plt.suptitle("Sample Original MNIST Images  |  24BAD081 - NIVASRI T", fontsize=13)
plt.tight_layout(); plt.show()

dist = pd.Series(y_train_full).value_counts().sort_index()
dist.index.name = "Digit"
print("Class distribution (training):")
print(dist.to_frame("Count").T.to_string())

In [ ]:
# ---- A.3 - A.5 Normalise to [0,1] and flatten to 784-d vectors ----
x_train_full = x_train_full.astype("float32") / 255.0
x_test       = x_test.astype("float32") / 255.0

INPUT_DIM = 28 * 28
x_train_full = x_train_full.reshape(-1, INPUT_DIM)
x_test       = x_test.reshape(-1, INPUT_DIM)

print("After normalisation -> min: %.1f, max: %.1f" % (x_train_full.min(), x_train_full.max()))
print("Flattened shapes    :", x_train_full.shape, x_test.shape)

# ---- A.6 Split into training / validation / testing ----
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, random_state=SEED, stratify=y_train_full)

print("\nTrain      :", x_train.shape)
print("Validation :", x_val.shape)
print("Test       :", x_test.shape)

# ---- A.7 Input data is also the target output (unsupervised reconstruction) ----
print("\nAutoencoder target = input  (x -> x).  Labels are NOT used for training.")

---
## PART B: IMPLEMENTING THE AUTOENCODER MODEL
Architecture: `784 → 128 → 64 → [32 bottleneck] → 64 → 128 → 784`

In [ ]:
# ---- B.1 - B.6 Build Encoder, bottleneck, Decoder and full Autoencoder ----
# Roll No: 24BAD081 | Name: NIVASRI T
LATENT_DIM = 32           # bottleneck size (784 -> 32 = ~24.5x compression)

# Input layer
input_img = keras.Input(shape=(INPUT_DIM,), name="input_image")

# Encoder
x = layers.Dense(128, activation='relu', name="enc_dense_1")(input_img)
x = layers.Dense(64,  activation='relu', name="enc_dense_2")(x)

# Bottleneck (fewer neurons than the input -> compact representation)
latent = layers.Dense(LATENT_DIM, activation='relu', name="bottleneck")(x)

# Decoder
x = layers.Dense(64,  activation='relu', name="dec_dense_1")(latent)
x = layers.Dense(128, activation='relu', name="dec_dense_2")(x)
output_img = layers.Dense(INPUT_DIM, activation='sigmoid', name="reconstruction")(x)

# Models
autoencoder = keras.Model(input_img, output_img, name="Autoencoder_24BAD081")
encoder     = keras.Model(input_img, latent,     name="Encoder_24BAD081")

# Standalone decoder (re-uses the trained decoder layers)
latent_input = keras.Input(shape=(LATENT_DIM,), name="latent_input")
d = autoencoder.get_layer("dec_dense_1")(latent_input)
d = autoencoder.get_layer("dec_dense_2")(d)
d = autoencoder.get_layer("reconstruction")(d)
decoder = keras.Model(latent_input, d, name="Decoder_24BAD081")

print("Compression ratio: %d -> %d  (%.1fx)" % (INPUT_DIM, LATENT_DIM, INPUT_DIM / LATENT_DIM))

In [ ]:
# ---- B.7 Compile: Adam optimiser + MSE reconstruction loss ----
autoencoder.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

# ---- B.8 Model architecture and summaries ----
autoencoder.summary()
encoder.summary()
decoder.summary()

In [ ]:
# Visual architecture diagram (saved as image; needs pydot/graphviz which Colab provides)
try:
    keras.utils.plot_model(autoencoder, to_file="autoencoder_24BAD081.png",
                           show_shapes=True, show_layer_names=True)
    from IPython.display import Image, display
    display(Image("autoencoder_24BAD081.png"))
except Exception as e:
    print("Architecture diagram skipped:", e)

---
## PART C: MODEL TRAINING AND PERFORMANCE EVALUATION

In [ ]:
# ---- C.1 - C.4 Train (input = target), validate, record losses ----
# Roll No: 24BAD081 | Name: NIVASRI T
EPOCHS     = 30
BATCH_SIZE = 256

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = autoencoder.fit(
    x_train, x_train,                       # target = input
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_val, x_val),
    callbacks=[early_stop],
    verbose=2
)
print("\nEpochs run:", len(history.history['loss']))
print("Final training loss   : %.5f" % history.history['loss'][-1])
print("Final validation loss : %.5f" % history.history['val_loss'][-1])

In [ ]:
# ---- C.5 - C.6 Evaluate on test data and compute reconstruction MSE ----
test_loss = autoencoder.evaluate(x_test, x_test, verbose=0)
x_test_recon = autoencoder.predict(x_test, verbose=0)

mse_overall = mean_squared_error(x_test, x_test_recon)
per_image_mse = np.mean((x_test - x_test_recon) ** 2, axis=1)

print("Test loss (Keras MSE)          : %.5f" % test_loss)
print("Reconstruction MSE (sklearn)   : %.5f" % mse_overall)
print("Per-image MSE  min / mean / max: %.5f / %.5f / %.5f" %
      (per_image_mse.min(), per_image_mse.mean(), per_image_mse.max()))

In [ ]:
# ---- C.7 Plot training and validation loss vs epochs ----
plt.figure(figsize=(8, 4.5))
plt.plot(history.history['loss'],     label='Training loss',   marker='o', ms=3)
plt.plot(history.history['val_loss'], label='Validation loss', marker='s', ms=3)
plt.title("Autoencoder Loss Curve  |  24BAD081 - NIVASRI T")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.legend(); plt.grid(alpha=.3)
plt.show()

pd.DataFrame(history.history).round(5).rename_axis("Epoch").tail(5)

In [ ]:
# ---- C.8 Compare original images with reconstructed outputs ----
n = 10
fig, axes = plt.subplots(2, n, figsize=(16, 3.8))
for i in range(n):
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap='gray');       axes[0, i].axis('off')
    axes[1, i].imshow(x_test_recon[i].reshape(28, 28), cmap='gray'); axes[1, i].axis('off')
    axes[1, i].set_title(f"MSE {per_image_mse[i]:.4f}", fontsize=7)
axes[0, 0].set_title("ORIGINAL", loc='left', fontsize=11, color='green')
plt.suptitle("Original (top) vs Reconstructed (bottom)  |  24BAD081 - NIVASRI T", fontsize=13)
plt.tight_layout(); plt.show()

---
## PART D: DIMENSIONALITY REDUCTION, FEATURE VISUALIZATION AND EXPERIMENT MANAGEMENT

> **Note:** The lab sheet line about *"new movie-review sentences / trained LSTM model"* is carried over from another experiment and does not apply to an image Autoencoder.  
> The equivalent here is passing **new unseen test images** through the trained Encoder/Decoder, which is done below.

In [ ]:
# ---- D.1 Pass NEW (unseen) test samples through the trained model ----
# Roll No: 24BAD081 | Name: NIVASRI T
new_idx     = np.random.RandomState(SEED).choice(len(x_test), 8, replace=False)
new_samples = x_test[new_idx]
new_recon   = autoencoder.predict(new_samples, verbose=0)

fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for i in range(8):
    axes[0, i].imshow(new_samples[i].reshape(28, 28), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(new_recon[i].reshape(28, 28), cmap='gray');   axes[1, i].axis('off')
    axes[0, i].set_title(f"Label {y_test[new_idx[i]]}", fontsize=8)
plt.suptitle("New Unseen Samples -> Autoencoder -> Reconstruction  |  24BAD081", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ---- D.2 - D.3 Latent representations and dimension comparison ----
z_test = encoder.predict(x_test, verbose=0)

summary = pd.DataFrame({
    "Representation": ["Original input", "Latent (bottleneck)"],
    "Shape":          [str(x_test.shape), str(z_test.shape)],
    "Features/sample":[x_test.shape[1], z_test.shape[1]],
    "Values stored":  [x_test.size, z_test.size],
})
print(summary.to_string(index=False))
print("\nDimensionality reduced by %.1f%%  (%d -> %d features)" %
      (100 * (1 - z_test.shape[1] / x_test.shape[1]), x_test.shape[1], z_test.shape[1]))
print("\nLatent vector of test image 0:\n", np.round(z_test[0], 2))

In [ ]:
# ---- D.4 Visualise the latent space (32-D -> 2-D for plotting) ----
n_vis = 3000
z_sub, y_sub = z_test[:n_vis], y_test[:n_vis]

# (a) PCA projection
z_pca = PCA(n_components=2, random_state=SEED).fit_transform(z_sub)
# (b) t-SNE projection
z_tsne = TSNE(n_components=2, random_state=SEED, init='pca', perplexity=30).fit_transform(z_sub)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, z2, title in zip(axes, [z_pca, z_tsne], ["PCA of Latent Space", "t-SNE of Latent Space"]):
    sc = ax.scatter(z2[:, 0], z2[:, 1], c=y_sub, cmap='tab10', s=8, alpha=.8)
    ax.set_title(f"{title}  |  24BAD081"); ax.grid(alpha=.3)
cbar = fig.colorbar(sc, ax=axes, ticks=range(10), fraction=0.025)
cbar.set_label("Digit class")
plt.show()

In [ ]:
# ---- D.5 Reconstruct selected test samples with the standalone Decoder ----
sel = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
recon_from_latent = decoder.predict(z_test[sel], verbose=0)

fig, axes = plt.subplots(3, 10, figsize=(16, 5.5))
for col, i in enumerate(sel):
    axes[0, col].imshow(x_test[i].reshape(28, 28), cmap='gray'); axes[0, col].axis('off')
    axes[1, col].imshow(z_test[i].reshape(4, 8), cmap='viridis'); axes[1, col].axis('off')
    axes[2, col].imshow(recon_from_latent[col].reshape(28, 28), cmap='gray'); axes[2, col].axis('off')
axes[0, 0].set_title("Original (784-D)", loc='left', fontsize=10)
axes[1, 0].set_title("Latent code (32-D)", loc='left', fontsize=10)
axes[2, 0].set_title("Decoder output", loc='left', fontsize=10)
plt.suptitle("Encoder -> Latent -> Decoder pipeline  |  24BAD081 - NIVASRI T", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ---- D.6 Autoencoder vs PCA with the SAME number of components (32) ----
pca32 = PCA(n_components=LATENT_DIM, random_state=SEED).fit(x_train)
x_pca_recon = pca32.inverse_transform(pca32.transform(x_test))
mse_pca = mean_squared_error(x_test, x_pca_recon)

comparison = pd.DataFrame({
    "Method": ["Autoencoder (non-linear)", "PCA (linear)"],
    "Latent dims": [LATENT_DIM, LATENT_DIM],
    "Test reconstruction MSE": [round(mse_overall, 5), round(mse_pca, 5)],
})
print(comparison.to_string(index=False))

fig, axes = plt.subplots(3, 8, figsize=(14, 5.5))
for i in range(8):
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(x_test_recon[i].reshape(28, 28), cmap='gray'); axes[1, i].axis('off')
    axes[2, i].imshow(x_pca_recon[i].reshape(28, 28), cmap='gray'); axes[2, i].axis('off')
axes[0, 0].set_title("Original", loc='left', fontsize=10)
axes[1, 0].set_title("Autoencoder", loc='left', fontsize=10)
axes[2, 0].set_title("PCA", loc='left', fontsize=10)
plt.suptitle("Autoencoder vs PCA Reconstruction  |  24BAD081", fontsize=12)
plt.tight_layout(); plt.show()

### Observations
- Each 784-pixel image was compressed to a 32-value latent vector (about 24.5x smaller), and the Decoder still reconstructs clearly recognizable digits, so the Encoder has learned the important features of the data.
- The training and validation loss curves fall together and stay close, which shows the model is learning without significant overfitting.
- The latent-space plots show digits of the same class forming clusters, even though the labels were never used in training. This confirms the Autoencoder learned meaningful features in an unsupervised way.
- Reconstructions are slightly blurry because MSE loss averages over uncertain pixels and the bottleneck discards fine detail. Unusual or thick-stroke digits have higher per-image MSE.
- With the same number of dimensions, the non-linear Autoencoder generally achieves a different (typically comparable or lower) reconstruction error than linear PCA, as shown in the comparison table.

## RESULT
A Dense Autoencoder was implemented and trained on MNIST. It compressed 784-dimensional images to a 32-dimensional latent representation and reconstructed them with low reconstruction MSE. The latent space was visualized with PCA and t-SNE, and the Autoencoder's performance was compared with PCA.

---
### Upload the notebook to GitHub (experiment management)
1. In Colab: **File → Save a copy in GitHub** (or **File → Download .ipynb** and upload manually).
2. Create/choose a repository (e.g. `24BAD081-DeepLearning-Lab`) and commit as `Expt 7 - Autoencoder`.
3. Add the repository link in your record.